# MLB Fantasy Baseball Data Notebook

A reference implementation for gathering hitting, pitching, and team stats from:
- **Baseball Savant** (Statcast) — advanced metrics, exit velocity, spin rate
- **FanGraphs** — season leaderboards, wRC+, FIP, xFIP
- **Baseball Reference** — traditional stats, team data

**Rate limit notes:**
- Baseball Reference: 6-second delay auto-handled by pybaseball; avoid looping individual player calls
- Baseball Savant: 25,000-row query limit; prefer aggregate endpoints over pitch-level data for 300-player use case
- FanGraphs: no official API; pybaseball handles scraping
- Caching: `pybaseball.cache.enable()` saves results locally to avoid repeat scrapes

## Section 1: Setup & Imports

In [109]:
# Install dependencies (run once)
# !pip install pybaseball mlb-statsapi pandas numpy

In [110]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np

import pybaseball
from pybaseball import (
    playerid_lookup,
    statcast_batter,
    statcast_pitcher,
    batting_stats,
    pitching_stats,
    batting_stats_bref,
    team_batting,
    team_pitching,
    statcast_batter_exitvelo_barrels,
    statcast_pitcher_pitch_arsenal,
    statcast_batter_percentile_ranks,
)

import statsapi  # MLB-StatsAPI

# Enable caching — results saved locally, avoids repeat scrapes
pybaseball.cache.enable()

print('Libraries loaded successfully.')
print(f'pybaseball version: {pybaseball.__version__}')

Libraries loaded successfully.
pybaseball version: 2.2.7


In [111]:
# ── CONFIG ──────────────────────────────────────────
SEASON = 2025  # change this to pull a different year
# ────────────────────────────────────────────────────

# Derived date strings used for pitch-level Statcast queries
SEASON_START = f'{SEASON}-03-01'
SEASON_END   = f'{SEASON}-11-01'

print(f'Config: SEASON={SEASON}, range {SEASON_START} → {SEASON_END}')

Config: SEASON=2025, range 2025-03-01 → 2025-11-01


## Section 2: Single Player Lookup

Demonstrates the simplest data pull for one player — useful for spot-checking and exploring available columns.

In [112]:
# Look up player IDs across data sources
player = playerid_lookup('Ohtani', 'Shohei')
print(player.columns.tolist())
player

['name_last', 'name_first', 'key_mlbam', 'key_retro', 'key_bbref', 'key_fangraphs', 'mlb_played_first', 'mlb_played_last']


,name_last,name_first,key_mlbam,key_retro,key_bbref,key_fangraphs,mlb_played_first,mlb_played_last
0,ohtani,shohei,660271,ohtas001,ohtansh01,19755,2018.0,2025.0


In [113]:
# Extract MLBAM (Baseball Savant) ID
ohtani_id = int(player.key_mlbam.iloc[0])
print(f'Ohtani MLBAM ID: {ohtani_id}')

Ohtani MLBAM ID: 660271


In [114]:
# Statcast pitch-level data for a single batter (Baseball Savant)
# Note: pitch-level data is large — use a short date range for exploration
ohtani_statcast = statcast_batter(
    start_dt=f'{SEASON}-04-01',
    end_dt=f'{SEASON}-04-30',
    player_id=ohtani_id,
)
print(f'Rows: {len(ohtani_statcast)}')
ohtani_statcast[['game_date', 'pitch_type', 'release_speed', 'launch_speed', 'launch_angle', 'events']].head(10)

Rows: 426


,game_date,pitch_type,release_speed,launch_speed,launch_angle,events
0,2025-04-30,FS,86.6,68.3,-46.0,force_out
1,2025-04-30,FS,87.1,NaN,NaN,NaN
2,2025-04-30,FS,86.2,NaN,NaN,NaN
3,2025-04-30,SI,93.6,NaN,NaN,NaN
4,2025-04-30,SL,83.8,NaN,NaN,walk
5,2025-04-30,SL,84.7,NaN,NaN,NaN
6,2025-04-30,FF,93.8,101.3,67.0,field_out
7,2025-04-30,SL,84.8,NaN,NaN,NaN
8,2025-04-30,SI,92.0,97.4,63.0,field_out
9,2025-04-30,SL,87.7,NaN,NaN,NaN


In [115]:
# Season FanGraphs stats for all batters, then filter to one player
all_batters_sample = batting_stats(SEASON, qual=0)
ohtani_fg = all_batters_sample[all_batters_sample['Name'] == 'Shohei Ohtani']
print(f'Total batters in dataset: {len(all_batters_sample)}')
ohtani_fg

Total batters in dataset: 1470


,IDfg,Season,Name,Team,Age,G,AB,PA,H,1B,...,maxEV,HardHit,HardHit%,Events,CStr%,CSW%,xBA,xSLG,xwOBA,L-WAR
6,19755,2025,Shohei Ohtani,LAD,30,158,611,727,172,83,...,120.0,250.0,0.584,428,0.147,0.289,0.274,0.649,0.425,7.2


## Section 3: Fangraphs | Bulk Season Stats (All Players)

Single API calls covering all qualified batters and pitchers — the most efficient approach for a 300-player use case.

In [116]:
# FanGraphs season stats — single call returns every player
# qual=0 includes all players regardless of plate appearances / innings
batters  = batting_stats(SEASON, qual=0)
pitchers = pitching_stats(SEASON, qual=0)

print(f'Batters:  {len(batters)} rows, {len(batters.columns)} columns')
print(f'Pitchers: {len(pitchers)} rows, {len(pitchers.columns)} columns')

Batters:  1470 rows, 320 columns
Pitchers: 873 rows, 393 columns


In [117]:
# Key fantasy hitting columns
HITTING_COLS = [
    'Name', 'Team', 'G', 'PA', 'HR', 'R', 'RBI', 'SB',
    'BB%', 'K%', 'AVG', 'OBP', 'SLG', 'wRC+',
    'xBA', 'xSLG', 'Barrel%',
]

# Only keep columns that exist in this season's data
hitting_cols_available = [c for c in HITTING_COLS if c in batters.columns]
missing_hitting = set(HITTING_COLS) - set(hitting_cols_available)
if missing_hitting:
    print(f'Note: columns not found this season: {missing_hitting}')

batters_display = batters[hitting_cols_available].copy()
batters_display.head(10)

,Name,Team,G,PA,HR,R,RBI,SB,BB%,K%,AVG,OBP,SLG,wRC+,xBA,xSLG,Barrel%
2,Aaron Judge,NYY,152,679,53,137,114,12,0.183,0.236,0.331,0.457,0.688,204.0,0.300,0.708,0.247
13,Cal Raleigh,SEA,159,705,60,110,125,14,0.138,0.267,0.247,0.359,0.589,161.0,0.231,0.547,0.195
53,Bobby Witt Jr.,KCR,157,687,23,99,88,38,0.071,0.182,0.295,0.351,0.501,130.0,0.285,0.508,0.125
6,Shohei Ohtani,LAD,158,727,55,146,102,20,0.150,0.257,0.282,0.392,0.622,172.0,0.274,0.649,0.234
26,Geraldo Perdomo,ARI,161,720,20,98,100,27,0.131,0.115,0.290,0.389,0.462,138.0,0.278,0.424,0.062
71,Trea Turner,PHI,141,639,15,94,69,36,0.067,0.167,0.304,0.355,0.457,125.0,0.270,0.410,0.058
25,Corbin Carroll,ARI,143,642,31,107,84,32,0.104,0.238,0.259,0.343,0.541,139.0,0.262,0.529,0.145
54,Jose Ramirez,CLE,158,673,30,103,85,44,0.098,0.110,0.283,0.360,0.503,133.0,0.284,0.446,0.070
76,Francisco Lindor,NYM,160,732,31,117,86,31,0.089,0.179,0.267,0.346,0.466,129.0,0.260,0.454,0.088
66,Fernando Tatis Jr.,SDP,155,691,25,111,71,32,0.129,0.187,0.268,0.368,0.446,131.0,0.272,0.488,0.109


In [118]:

# ── ALL AVAILABLE BATTING COLUMNS (FanGraphs via pybaseball) ────────────────
# Reference cell — add any of these to HITTING_COLS below.
# batting_stats() returns ~320 columns total; grouped by category here.

# ── IDENTITY ─────────────────────────────────────────────────────────────────
# IDfg          FanGraphs player ID
# Season        Season year
# Name          Player name
# Team          Team abbreviation
# Age           Player age

# ── COUNTING STATS ───────────────────────────────────────────────────────────
# G             Games played
# AB            At-bats
# PA            Plate appearances
# H             Hits
# 1B            Singles
# 2B            Doubles
# 3B            Triples
# HR            Home runs
# R             Runs scored
# RBI           Runs batted in
# BB            Walks
# IBB           Intentional walks
# SO            Strikeouts
# HBP           Hit by pitch
# SF            Sacrifice flies
# SH            Sacrifice hits
# GDP           Grounded into double play
# SB            Stolen bases
# CS            Caught stealing

# ── RATE / TRADITIONAL ───────────────────────────────────────────────────────
# AVG           Batting average
# OBP           On-base percentage
# SLG           Slugging percentage
# OPS           On-base plus slugging
# ISO           Isolated power (SLG - AVG)
# BABIP         Batting average on balls in play
# BB%           Walk rate
# K%            Strikeout rate
# BB/K          Walk-to-strikeout ratio

# ── ADVANCED / SABERMETRIC ───────────────────────────────────────────────────
# wOBA          Weighted on-base average
# wRAA          Weighted runs above average
# wRC           Weighted runs created
# wRC+          Weighted runs created plus (league/park adjusted, 100 = avg)
# Spd           Speed score

# ── STATCAST / xSTATS (Baseball Savant) ──────────────────────────────────────
# EV            Average exit velocity (mph)
# maxEV         Max exit velocity
# LA            Average launch angle
# Barrel        Barrel count
# Barrel%       Barrel rate per PA
# HardHit       Hard-hit ball count (EV >= 95 mph)
# HardHit%      Hard-hit rate
# xBA           Expected batting average
# xSLG          Expected slugging percentage
# xwOBA         Expected weighted on-base average
# xwOBAcon      Expected wOBA on contact
# xEV           Expected exit velocity
# xBACON        Expected BA on contact

# ── BATTED BALL ───────────────────────────────────────────────────────────────
# GB/FB         Ground ball to fly ball ratio
# LD%           Line drive rate
# GB%           Ground ball rate
# FB%           Fly ball rate
# IFFB%         Infield fly ball rate
# HR/FB         Home run to fly ball rate
# IFH           Infield hits
# IFH%          Infield hit rate
# BUH           Bunt hits
# BUH%          Bunt hit rate
# Pull%         Pull rate
# Cent%         Center rate
# Oppo%         Opposite field rate
# Soft%         Soft contact rate
# Med%          Medium contact rate
# Hard%         Hard contact rate

# ── PLATE DISCIPLINE ─────────────────────────────────────────────────────────
# O-Swing%      Chase rate (swing % on pitches outside zone)
# Z-Swing%      Swing % on pitches inside zone
# Swing%        Overall swing rate
# O-Contact%    Contact % on pitches outside zone
# Z-Contact%    Contact % on pitches inside zone
# Contact%      Overall contact rate
# Zone%         % of pitches seen in zone
# F-Strike%     First-pitch strike rate (seen by batter)
# SwStr%        Swinging strike rate
# CStr%         Called strike rate
# CSW%          Called + swinging strike rate

# ── WIN PROBABILITY ───────────────────────────────────────────────────────────
# WPA           Win probability added
# -WPA          Negative WPA (clutch failures)
# +WPA          Positive WPA (clutch successes)
# RE24          Run expectancy based on 24 base-out states
# REW           RE24 converted to wins
# pLI           Average leverage index
# phLI          Pinch-hit leverage index
# PH            Pinch-hit appearances
# WPA/LI        Clutch-neutral WPA (WPA scaled by leverage)
# Clutch        How much better/worse in high-leverage vs overall

# ── VALUE ─────────────────────────────────────────────────────────────────────
# RAR           Runs above replacement
# WAR           Wins above replacement (FanGraphs version = fWAR)
# Dol           Dollar value above replacement
# L-WAR         Combined batter + baserunning + positional WAR

# ── PITCH TYPE RATES (% of pitches seen of each type) ─────────────────────────
# FA%   FC%   FS%   FO%   SI%   SL%   CU%   KC%   EP%   CH%   SC%   KN%   UN%
# (FA=4-seam, FC=cutter, FS=splitter, SI=sinker, SL=slider,
#  CU=curveball, KC=knuckle-curve, CH=changeup, KN=knuckleball)

# ── PITCH VALUES (runs above average per 100 pitches of that type) ─────────────
# wFA   wFC   wFS   wFO   wSI   wSL   wCU   wKC   wEP   wCH   wSC   wKN
# wFA/C wFC/C wFS/C wFO/C wSI/C wSL/C wCU/C wKC/C wEP/C wCH/C wSC/C wKN/C

# ── SPRINT SPEED ──────────────────────────────────────────────────────────────
# Sprint_Speed  Feet per second on competitive runs (Baseball Savant)

# ─────────────────────────────────────────────────────────────────────────────
# To see every column in your current dataset: print(batters.columns.tolist())
# ─────────────────────────────────────────────────────────────────────────────


In [119]:

# ── ALL AVAILABLE PITCHING COLUMNS (FanGraphs via pybaseball) ───────────────
# Reference cell — add any of these to PITCHING_COLS below.
# pitching_stats() returns ~393 columns total; grouped by category here.

# ── IDENTITY ─────────────────────────────────────────────────────────────────
# IDfg          FanGraphs player ID
# Season        Season year
# Name          Player name
# Team          Team abbreviation
# Age           Player age

# ── COUNTING STATS ───────────────────────────────────────────────────────────
# G             Games pitched
# GS            Games started
# CG            Complete games
# ShO           Shutouts
# SV            Saves
# BS            Blown saves
# HLD           Holds
# IP            Innings pitched
# TBF           Total batters faced
# H             Hits allowed
# R             Runs allowed
# ER            Earned runs allowed
# HR            Home runs allowed
# BB            Walks issued
# IBB           Intentional walks issued
# HBP           Hit batters
# WP            Wild pitches
# BK            Balks
# SO            Strikeouts

# ── TRADITIONAL RATES ────────────────────────────────────────────────────────
# W             Wins
# L             Losses
# ERA           Earned run average
# WHIP          Walks + hits per inning pitched
# AVG           Batting average against
# H/9           Hits per 9 innings
# HR/9          Home runs per 9 innings
# K/9           Strikeouts per 9 innings
# BB/9          Walks per 9 innings
# K/BB          Strikeout-to-walk ratio
# K%            Strikeout rate
# BB%           Walk rate
# K-BB%         Strikeout minus walk rate
# BABIP         Batting average on balls in play against
# LOB%          Left-on-base percentage (strand rate)
# ERA-          ERA relative to league average (100 = avg, lower is better)
# FIP-          FIP relative to league average

# ── ADVANCED / SABERMETRIC ───────────────────────────────────────────────────
# FIP           Fielding independent pitching
# xFIP          Expected FIP (normalizes HR/FB rate to league average)
# SIERA         Skill-interactive ERA (batted ball and K/BB weighted)
# E-F           ERA minus FIP (positive = ERA inflated vs FIP)
# xERA          Expected ERA (Statcast-based)
# RS            Run support
# RS/9          Run support per 9 innings

# ── STATCAST / xSTATS (Baseball Savant) ──────────────────────────────────────
# EV            Average exit velocity allowed
# maxEV         Max exit velocity allowed
# LA            Average launch angle allowed
# Barrel        Barrels allowed count
# Barrel%       Barrel rate per PA against
# HardHit       Hard-hit balls allowed
# HardHit%      Hard-hit rate against
# xBA           Expected batting average against
# xSLG          Expected slugging against
# xwOBA         Expected wOBA against
# xwOBAcon      Expected wOBA on contact against
# xFIP          Statcast-based expected FIP (sometimes listed separately)

# ── BATTED BALL ───────────────────────────────────────────────────────────────
# GB/FB         Ground ball to fly ball ratio (higher = more ground balls)
# LD%           Line drive rate against
# GB%           Ground ball rate against
# FB%           Fly ball rate against
# IFFB%         Infield fly ball rate against
# HR/FB         Home run to fly ball rate against
# IFH           Infield hits allowed
# IFH%          Infield hit rate against
# BUH           Bunt hits allowed
# BUH%          Bunt hit rate against
# Pull%         Pull rate of batters faced
# Cent%         Center rate of batters faced
# Oppo%         Opposite field rate of batters faced
# Soft%         Soft contact rate allowed
# Med%          Medium contact rate allowed
# Hard%         Hard contact rate allowed

# ── PLATE DISCIPLINE ─────────────────────────────────────────────────────────
# O-Swing%      Opponent chase rate
# Z-Swing%      Opponent swing % in zone
# Swing%        Opponent overall swing rate
# O-Contact%    Opponent contact % outside zone
# Z-Contact%    Opponent contact % in zone
# Contact%      Opponent overall contact rate
# Zone%         % of pitches thrown in zone
# F-Strike%     First-pitch strike rate (thrown)
# SwStr%        Swinging strike rate
# CStr%         Called strike rate
# CSW%          Called + swinging strike rate

# ── WIN PROBABILITY ───────────────────────────────────────────────────────────
# WPA           Win probability added
# -WPA          Negative WPA
# +WPA          Positive WPA
# RE24          Run expectancy above average
# REW           RE24 in wins
# pLI           Average leverage index when entering game
# inLI          Leverage index at start of inning
# gmLI          Leverage index at start of game appearance
# exLI          Expected leverage index
# Pulls         Times pulled from game
# WPA/LI        Clutch-neutral WPA
# Clutch        Performance in high-leverage vs overall

# ── VALUE ─────────────────────────────────────────────────────────────────────
# RAR           Runs above replacement
# WAR           Wins above replacement (FanGraphs fWAR)
# Dol           Dollar value above replacement

# ── PITCH TYPE RATES (% of pitches thrown of each type) ─────────────────────
# FA%   FC%   FS%   FO%   SI%   SL%   CU%   KC%   EP%   CH%   SC%   KN%   UN%
# (FA=4-seam, FC=cutter, FS=splitter, SI=sinker, SL=slider,
#  CU=curveball, KC=knuckle-curve, CH=changeup, KN=knuckleball)

# ── PITCH VELOCITY (avg mph per pitch type) ───────────────────────────────────
# vFA   vFC   vFS   vFO   vSI   vSL   vCU   vKC   vEP   vCH   vSC   vKN

# ── PITCH MOVEMENT (inches, pfx coords) ──────────────────────────────────────
# FA-X  FC-X  FS-X  SI-X  SL-X  CU-X  KC-X  CH-X       (horizontal break)
# FA-Z  FC-Z  FS-Z  SI-Z  SL-Z  CU-Z  KC-Z  CH-Z       (vertical break)

# ── PITCH VALUES (runs above avg per 100 pitches thrown) ──────────────────────
# wFA   wFC   wFS   wFO   wSI   wSL   wCU   wKC   wEP   wCH   wSC   wKN
# wFA/C wFC/C wFS/C wFO/C wSI/C wSL/C wCU/C wKC/C wEP/C wCH/C wSC/C wKN/C

# ─────────────────────────────────────────────────────────────────────────────
# To see every column in your current dataset: print(pitchers.columns.tolist())
# ─────────────────────────────────────────────────────────────────────────────


In [120]:
# Key fantasy pitching columns
PITCHING_COLS = [
    'Name', 'Team', 'G', 'GS', 'IP', 'W', 'SV',
    'K/9', 'BB/9', 'ERA', 'WHIP', 'FIP', 'xFIP', 'K%',
]

pitching_cols_available = [c for c in PITCHING_COLS if c in pitchers.columns]
missing_pitching = set(PITCHING_COLS) - set(pitching_cols_available)
if missing_pitching:
    print(f'Note: columns not found this season: {missing_pitching}')

pitchers_display = pitchers[pitching_cols_available].copy()
pitchers_display.head(10)

,Name,Team,G,GS,IP,W,SV,K/9,BB/9,ERA,WHIP,FIP,xFIP,K%
99,Tarik Skubal,DET,31,31,195.1,13,0,11.10,1.52,2.21,0.89,2.45,2.66,0.322
78,Paul Skenes,PIT,32,32,187.2,10,0,10.36,2.01,1.97,0.95,2.36,3.03,0.295
138,Cristopher Sanchez,PHI,32,32,202.0,13,0,9.45,1.96,2.50,1.06,2.55,2.77,0.263
145,Garrett Crochet,BOS,32,32,205.1,18,0,11.18,2.02,2.59,1.03,2.89,2.64,0.313
253,Logan Webb,SFG,34,34,207.0,15,0,9.74,2.00,3.22,1.24,2.60,2.78,0.262
365,Jesus Luzardo,PHI,32,32,183.2,15,0,10.58,2.79,3.92,1.22,2.90,3.25,0.285
136,Yoshinobu Yamamoto,LAD,30,30,173.2,12,0,10.42,3.06,2.49,0.99,2.94,3.05,0.294
181,Max Fried,NYY,32,32,195.1,19,0,8.71,2.35,2.86,1.10,3.07,3.41,0.236
121,Hunter Brown,HOU,31,31,185.1,12,0,10.00,2.77,2.43,1.03,3.14,3.19,0.283
316,Kevin Gausman,TOR,32,32,193.0,10,0,8.81,2.33,3.59,1.06,3.41,3.77,0.244


## Section 4: Statcast / Baseball Savant Data

Season-level aggregate endpoints — much smaller than pitch-level data and ideal for the 300-player use case.

In [121]:
# Exit velocity and barrel rate aggregates (Baseball Savant)
ev_barrels = statcast_batter_exitvelo_barrels(SEASON)
print(f'EV/Barrels: {len(ev_barrels)} rows, {len(ev_barrels.columns)} columns')
print(f'Columns: {ev_barrels.columns.tolist()}')

# Identify available name columns — API column names vary by season/version
preview_cols = ['avg_hit_speed', 'max_hit_speed', 'brl_pa', 'brl_percent']
if 'last_name' in ev_barrels.columns and 'first_name' in ev_barrels.columns:
    preview_cols = ['last_name', 'first_name'] + preview_cols
elif 'last_name, first_name' in ev_barrels.columns:
    preview_cols = ['last_name, first_name'] + preview_cols
elif 'player_name' in ev_barrels.columns:
    preview_cols = ['player_name'] + preview_cols

preview_cols_available = [c for c in preview_cols if c in ev_barrels.columns]
ev_barrels[preview_cols_available].head(10)

EV/Barrels: 251 rows, 18 columns
Columns: ['last_name, first_name', 'player_id', 'attempts', 'avg_hit_angle', 'anglesweetspotpercent', 'max_hit_speed', 'avg_hit_speed', 'ev50', 'fbld', 'gb', 'max_distance', 'avg_distance', 'avg_hr_distance', 'ev95plus', 'ev95percent', 'barrels', 'brl_percent', 'brl_pa']


,"last_name, first_name",avg_hit_speed,max_hit_speed,brl_pa,brl_percent
0,"Arraez, Luis",86.1,107.8,1.0,1.1
1,"Kwan, Steven",86.2,104.2,1.6,1.9
2,"Hoerner, Nico",86.7,108.1,2.0,2.3
3,"Perdomo, Geraldo",87.6,108.2,4.6,6.2
4,"Betts, Mookie",89.1,108.4,4.4,5.5
5,"Ramírez, José",88.9,111.6,5.5,7.0
6,"Lindor, Francisco",90.5,112.9,6.3,8.8
7,"Garcia, Maikel",91.3,110.3,4.4,5.6
8,"Pasquantino, Vinnie",90.9,114.4,8.2,10.8
9,"Albies, Ozzie",87.5,109.9,3.7,4.9


In [122]:
# Pitcher spin rate aggregates (Baseball Savant)
# statcast_pitcher_pitch_arsenal returns per-pitcher avg spin rates for each pitch type
spin = statcast_pitcher_pitch_arsenal(SEASON, minP=100, arsenal_type='avg_spin')
print(f'Spin rates: {len(spin)} rows, {len(spin.columns)} columns')
spin.head(10)

Spin rates: 723 rows, 12 columns


,"last_name, first_name",pitcher,ff_avg_spin,si_avg_spin,fc_avg_spin,sl_avg_spin,ch_avg_spin,cu_avg_spin,fs_avg_spin,kn_avg_spin,st_avg_spin,sv_avg_spin
0,"Webb, Logan",657277,2118.0,1959.0,2225.0,NaN,1502.0,NaN,NaN,NaN,2543.0,NaN
1,"Rodón, Carlos",607074,2347.0,2256.0,2383.0,2633.0,1650.0,2717.0,NaN,NaN,NaN,NaN
2,"Crochet, Garrett",676979,2460.0,2399.0,2596.0,NaN,1711.0,NaN,NaN,NaN,2378.0,NaN
3,"Gallen, Zac",668678,2322.0,2139.0,2338.0,2379.0,1518.0,2441.0,NaN,NaN,NaN,NaN
4,"Fried, Max",608331,2211.0,2132.0,2302.0,2538.0,1659.0,2920.0,NaN,NaN,2838.0,NaN
5,"Kikuchi, Yusei",579328,2185.0,2157.0,NaN,2295.0,1541.0,2526.0,NaN,NaN,2389.0,NaN
6,"Peralta, Freddy",642547,2441.0,NaN,NaN,2453.0,1929.0,2331.0,NaN,NaN,NaN,NaN
7,"Ray, Robbie",592662,2360.0,NaN,NaN,2269.0,1650.0,2185.0,NaN,NaN,2206.0,NaN
8,"Cease, Dylan",656302,2550.0,2477.0,NaN,2763.0,1681.0,2746.0,NaN,NaN,2914.0,NaN
9,"Gausman, Kevin",592332,2260.0,2280.0,NaN,2334.0,NaN,NaN,1691.0,NaN,NaN,NaN


In [123]:
# Batter percentile ranks (Baseball Savant)
pct_ranks = statcast_batter_percentile_ranks(SEASON)
print(f'Percentile ranks: {len(pct_ranks)} rows, {len(pct_ranks.columns)} columns')
pct_ranks.head(10)

Percentile ranks: 617 rows, 23 columns


,player_name,player_id,year,xwoba,xba,xslg,xiso,xobp,brl,brl_percent,...,k_percent,bb_percent,whiff_percent,chase_percent,arm_strength,sprint_speed,oaa,bat_speed,squared_up_rate,swing_length
0,"Campero, Gustavo",672569,2025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,86.0,91,NaN,NaN,NaN,NaN
1,"Harris, Brett",695391,2025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,58,NaN,NaN,NaN,NaN
2,"Valera, George",671655,2025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,18,NaN,NaN,NaN,NaN
3,"Campusano, Luis",669134,2025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,6,NaN,NaN,NaN,NaN
4,"Walton, Donovan",622268,2025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,16,NaN,NaN,NaN,NaN
5,"Gorman, Nolan",669357,2025,16.0,1.0,29.0,62.0,12.0,24.0,51.0,...,1.0,84.0,2.0,50.0,37.0,15,5.0,61.0,4.0,18.0
6,"Baldwin, Brooks",681460,2025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,55.0,82,2.0,NaN,NaN,NaN
7,"Sheets, Gavin",657757,2025,73.0,73.0,68.0,62.0,65.0,59.0,54.0,...,58.0,48.0,61.0,27.0,61.0,38,NaN,86.0,45.0,12.0
8,"Contreras, Willson",575929,2025,86.0,62.0,84.0,85.0,78.0,84.0,86.0,...,24.0,44.0,23.0,35.0,76.0,49,91.0,95.0,10.0,3.0
9,"Peraza, Oswald",672724,2025,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,81.0,93,77.0,NaN,NaN,NaN


In [124]:
# Baseball Reference season batting stats (alternative source)
bref_bat = batting_stats_bref(SEASON)
print(f'Baseball Reference batting: {len(bref_bat)} rows')
bref_bat.head(5)

Baseball Reference batting: 675 rows


,Name,Age,#days,Lev,Tm,G,PA,AB,R,H,...,SH,SF,GDP,SB,CS,BA,OBP,SLG,OPS,mlbID
1,CJ Abrams,24,150,Maj-NL,Washington,144,635,580,92,149,...,1,3,5,31,3,0.257,0.315,0.433,0.748,682928
2,Wilyer Abreu,26,146,Maj-AL,Boston,116,422,378,53,92,...,1,3,8,6,3,0.243,0.314,0.463,0.777,677800
3,Maximo Acosta,22,164,Maj-NL,Miami,19,61,54,7,11,...,0,0,0,1,2,0.204,0.295,0.389,0.684,691185
4,Luisangel Acu\xc3\xb1a,23,151,Maj-NL,New York,95,193,175,30,41,...,2,1,6,16,1,0.234,0.293,0.274,0.567,682668
5,Ronald Acu\xc3\xb1a Jr.,27,150,Maj-NL,Atlanta,95,412,338,74,98,...,0,0,5,9,1,0.290,0.417,0.518,0.935,660670


## Section 5: Top 300 Fantasy Players — Full Pipeline

Combines FanGraphs leaderboards with Statcast aggregates, then filters to a ranked top-300 list.

In [125]:
# ── Step 1: Bulk season stats (already pulled above, reuse if cached) ──
# batters  = batting_stats(SEASON, qual=0)   # already loaded in Section 3
# pitchers = pitching_stats(SEASON, qual=0)  # already loaded in Section 3

# ── Step 2: Statcast aggregates (already pulled above) ──
# ev_barrels = statcast_batter_exitvelo_barrels(SEASON)  # loaded in Section 4
# spin       = statcast_pitcher_pitch_arsenal(SEASON, minP=100, arsenal_type='avg_spin')  # loaded in Section 4

print('All source dataframes ready.')
print(f'  batters:    {len(batters)} rows')
print(f'  pitchers:   {len(pitchers)} rows')
print(f'  ev_barrels: {len(ev_barrels)} rows')
print(f'  spin:       {len(spin)} rows')

All source dataframes ready.
  batters:    1470 rows
  pitchers:   873 rows
  ev_barrels: 251 rows
  spin:       723 rows


In [126]:
# ── Step 3a: Merge batter Statcast onto FanGraphs hitting ──
# Build a 'Name' merge key — handle different column naming across API versions
if 'first_name' in ev_barrels.columns and 'last_name' in ev_barrels.columns:
    ev_barrels['Name'] = (
        ev_barrels['first_name'].str.strip() + ' ' +
        ev_barrels['last_name'].str.strip()
    )
elif 'last_name, first_name' in ev_barrels.columns:
    # Combined column in "Last, First" format — split and reverse to "First Last"
    ev_barrels['Name'] = ev_barrels['last_name, first_name'].apply(
        lambda x: ' '.join(reversed(x.split(', '))) if isinstance(x, str) and ', ' in x else x
    )
elif 'player_name' in ev_barrels.columns:
    # Some seasons return 'player_name' in "Last, First" format
    ev_barrels['Name'] = ev_barrels['player_name'].apply(
        lambda x: ' '.join(reversed(x.split(', '))) if isinstance(x, str) and ',' in x else x
    )
else:
    raise ValueError(f'Cannot find name columns in ev_barrels. Available: {ev_barrels.columns.tolist()}')

# Columns to bring in from Statcast (avoid duplicating Name)
EV_COLS = ['Name', 'avg_hit_speed', 'max_hit_speed', 'brl_pa', 'brl_percent', 'player_id']
ev_cols_available = [c for c in EV_COLS if c in ev_barrels.columns]

batters_merged = batters.merge(
    ev_barrels[ev_cols_available],
    on='Name',
    how='left',
)

print(f'Merged batter dataset: {len(batters_merged)} rows, {len(batters_merged.columns)} columns')

Merged batter dataset: 1470 rows, 325 columns


In [127]:
# ── Step 3b: Merge pitcher Statcast onto FanGraphs pitching ──
if 'last_name' in spin.columns and 'first_name' in spin.columns:
    spin['Name'] = (
        spin['first_name'].str.strip() + ' ' +
        spin['last_name'].str.strip()
    )
elif 'last_name, first_name' in spin.columns:
    # Combined column in "Last, First" format — split and reverse to "First Last"
    spin['Name'] = spin['last_name, first_name'].apply(
        lambda x: ' '.join(reversed(x.split(', '))) if isinstance(x, str) and ', ' in x else x
    )
elif 'player_name' in spin.columns:
    # Some seasons return 'player_name' in "Last, First" format
    spin['Name'] = spin['player_name'].apply(
        lambda x: ' '.join(reversed(x.split(', '))) if isinstance(x, str) and ',' in x else x
    )

# Drop raw name column(s) before merge; keep pitcher ID if present
SPIN_EXCLUDE = {'last_name, first_name', 'last_name', 'first_name', 'player_name'}
SPIN_COLS = [c for c in spin.columns if c not in SPIN_EXCLUDE]

pitchers_merged = pitchers.merge(
    spin[SPIN_COLS],
    on='Name',
    how='left',
)

print(f'Merged pitcher dataset: {len(pitchers_merged)} rows, {len(pitchers_merged.columns)} columns')

Merged pitcher dataset: 873 rows, 404 columns


In [128]:
# ── Step 4: Filter and rank to top 300 ──

# Batters: minimum 100 PA; rank by wRC+ (or PA if wRC+ unavailable)
MIN_PA = 100
rank_col_bat = 'wRC+' if 'wRC+' in batters_merged.columns else 'PA'

qualified_batters = (
    batters_merged[batters_merged['PA'] >= MIN_PA]
    .sort_values(rank_col_bat, ascending=False)
    .reset_index(drop=True)
)
qualified_batters.index += 1  # 1-based rank
qualified_batters.index.name = 'rank'

# Pitchers: minimum 20 IP; rank by FIP (lower is better) — use ascending
MIN_IP = 20
rank_col_pit = 'FIP' if 'FIP' in pitchers_merged.columns else 'IP'
pit_ascending = rank_col_pit == 'FIP'

qualified_pitchers = (
    pitchers_merged[pitchers_merged['IP'] >= MIN_IP]
    .sort_values(rank_col_pit, ascending=pit_ascending)
    .reset_index(drop=True)
)
qualified_pitchers.index += 1
qualified_pitchers.index.name = 'rank'

# Combine: take up to 200 batters + 100 pitchers = 300 total
top_batters  = qualified_batters.head(200)
top_pitchers = qualified_pitchers.head(100)

top_batters['player_type']  = 'batter'
top_pitchers['player_type'] = 'pitcher'

top_300 = pd.concat([top_batters, top_pitchers], ignore_index=True)
top_300.index += 1
top_300.index.name = 'overall_rank'

print(f'Top 300 breakdown:')
print(f'  Batters:  {len(top_batters)}')
print(f'  Pitchers: {len(top_pitchers)}')
print(f'  Total:    {len(top_300)}')

Top 300 breakdown:
  Batters:  200
  Pitchers: 100
  Total:    300


In [129]:
# Preview top 300 — key columns
preview_cols = ['Name', 'Team', 'player_type', 'G', 'PA', 'HR', 'RBI', 'SB', 'AVG', 'wRC+', 'IP', 'ERA', 'FIP']
preview_cols_available = [c for c in preview_cols if c in top_300.columns]
top_300[preview_cols_available].head(20)

,Name,Team,player_type,G,PA,HR,RBI,SB,AVG,wRC+,IP,ERA,FIP
overall_rank,,,,,,,,,,,,,
1,Aaron Judge,NYY,batter,152,679.0,53,114.0,12.0,0.331,204.0,NaN,NaN,NaN
2,Shohei Ohtani,LAD,batter,158,727.0,55,102.0,20.0,0.282,172.0,NaN,NaN,NaN
3,Nick Kurtz,ATH,batter,117,489.0,36,86.0,2.0,0.290,170.0,NaN,NaN,NaN
4,George Springer,TOR,batter,140,586.0,32,84.0,18.0,0.309,166.0,NaN,NaN,NaN
5,Ronald Acuna Jr.,ATL,batter,95,412.0,21,42.0,9.0,0.290,161.0,NaN,NaN,NaN
6,Cal Raleigh,SEA,batter,159,705.0,60,125.0,14.0,0.247,161.0,NaN,NaN,NaN
7,Jahmai Jones,DET,batter,72,150.0,7,23.0,2.0,0.287,159.0,NaN,NaN,NaN
8,Giancarlo Stanton,NYY,batter,77,281.0,24,66.0,0.0,0.273,158.0,NaN,NaN,NaN
9,Juan Soto,NYM,batter,160,715.0,43,105.0,38.0,0.263,156.0,NaN,NaN,NaN


In [130]:
# Check for missing values in key stat columns
key_bat_stats  = [c for c in ['HR', 'RBI', 'SB', 'AVG', 'OBP', 'SLG'] if c in top_300.columns]
key_pit_stats  = [c for c in ['ERA', 'WHIP', 'FIP', 'K/9'] if c in top_300.columns]
all_key_stats  = key_bat_stats + key_pit_stats

print('Missing values in key stat columns:')
print(top_300[all_key_stats].isnull().sum())

Missing values in key stat columns:
HR        0
RBI     100
SB      100
AVG       0
OBP     100
SLG     100
ERA     200
WHIP    200
FIP     200
K/9     200
dtype: int64


In [131]:
# ── Step 5: Export to CSV ──
output_path = f'top_300_fantasy_{SEASON}.csv'
top_300.to_csv(output_path, index=True)
print(f'Exported {len(top_300)} players to {output_path}')

Exported 300 players to top_300_fantasy_2025.csv


## Section 6: Team-Level Data

Season aggregates for all 30 MLB teams.

In [132]:
# Team batting (FanGraphs)
team_bat = team_batting(SEASON)
print(f'Team batting: {len(team_bat)} teams, {len(team_bat.columns)} columns')
assert len(team_bat) == 30, f'Expected 30 teams, got {len(team_bat)}'
team_bat[['Team', 'G', 'PA', 'HR', 'R', 'RBI', 'SB', 'AVG', 'OBP', 'SLG', 'wRC+']].sort_values('wRC+', ascending=False)

Team batting: 30 teams, 319 columns


,Team,G,PA,HR,R,RBI,SB,AVG,OBP,SLG,wRC+
0,NYY,2401,6235,274,849,820,134,0.251,0.332,0.455,119
1,LAD,2438,6187,244,825,791,88,0.253,0.327,0.441,113
10,SEA,2424,6199,238,766,734,161,0.244,0.320,0.420,113
2,TOR,2481,6180,191,798,771,77,0.265,0.333,0.427,112
5,NYM,2402,6178,224,766,746,147,0.249,0.326,0.427,112
6,CHC,2330,6162,223,793,771,161,0.249,0.320,0.430,110
3,PHI,2300,6166,212,778,753,124,0.258,0.328,0.431,109
4,ARI,2332,6210,214,791,768,121,0.251,0.325,0.433,109
9,MIL,2413,6227,166,806,750,164,0.258,0.332,0.403,107
7,ATH,2367,6151,219,733,709,80,0.253,0.318,0.431,105


In [133]:
# Team pitching (FanGraphs)
team_pit = team_pitching(SEASON)
print(f'Team pitching: {len(team_pit)} teams, {len(team_pit.columns)} columns')
assert len(team_pit) == 30, f'Expected 30 teams, got {len(team_pit)}'
team_pit[['Team', 'G', 'IP', 'ERA', 'WHIP', 'FIP', 'K/9', 'BB/9']].sort_values('ERA')

Team pitching: 30 teams, 392 columns


,Team,G,IP,ERA,WHIP,FIP,K/9,BB/9
0,TEX,685,1443.0,3.49,1.18,3.89,8.38,2.89
1,MIL,712,1442.0,3.59,1.23,3.91,8.94,3.33
2,SDP,739,1433.0,3.64,1.21,4.02,8.95,3.36
3,CLE,693,1442.0,3.70,1.26,3.95,8.62,3.29
4,BOS,698,1448.1,3.72,1.29,3.98,8.46,3.29
5,KCR,709,1436.2,3.73,1.24,4.01,7.96,2.99
6,PIT,672,1430.2,3.76,1.22,3.83,8.27,2.98
7,PHI,667,1440.1,3.79,1.23,3.70,9.19,2.72
8,CHC,685,1435.0,3.81,1.18,4.16,7.93,2.54
9,SFG,677,1433.2,3.84,1.30,3.74,8.52,3.18


## Section 7: Verification Checks

Spot-checks to confirm data quality before using the pipeline.

In [134]:
# 1. Confirm all DataFrames are non-empty
checks = {
    'batters (FanGraphs)':       batters,
    'pitchers (FanGraphs)':      pitchers,
    'ev_barrels (Savant)':       ev_barrels,
    'spin (Savant)':             spin,
    'pct_ranks (Savant)':        pct_ranks,
    'bref_bat (Bref)':           bref_bat,
    'team_bat':                  team_bat,
    'team_pit':                  team_pit,
    'top_300':                   top_300,
}

print('DataFrame sizes:')
for name, df in checks.items():
    status = 'OK' if len(df) > 0 else 'EMPTY'
    print(f'  [{status}] {name}: {len(df)} rows')

DataFrame sizes:
  [OK] batters (FanGraphs): 1470 rows
  [OK] pitchers (FanGraphs): 873 rows
  [OK] ev_barrels (Savant): 251 rows
  [OK] spin (Savant): 723 rows
  [OK] pct_ranks (Savant): 617 rows
  [OK] bref_bat (Bref): 675 rows
  [OK] team_bat: 30 rows
  [OK] team_pit: 30 rows
  [OK] top_300: 300 rows


In [135]:
# 2. Spot-check Ohtani across sources
print('=== Ohtani spot-check ===\n')

ohtani_fg_check = batters[batters['Name'] == 'Shohei Ohtani']
if not ohtani_fg_check.empty:
    cols = [c for c in ['Name', 'Team', 'G', 'PA', 'HR', 'RBI', 'AVG', 'wRC+'] if c in ohtani_fg_check.columns]
    print('FanGraphs:')
    print(ohtani_fg_check[cols].to_string(index=False))
else:
    print('FanGraphs: Ohtani not found (may not have batted this season)')

print()
ohtani_ev = ev_barrels[ev_barrels['Name'] == 'Shohei Ohtani'] if 'Name' in ev_barrels.columns else pd.DataFrame()
if not ohtani_ev.empty:
    ev_cols = [c for c in ['Name', 'avg_hit_speed', 'max_hit_speed', 'brl_percent'] if c in ohtani_ev.columns]
    print('Statcast EV/Barrels:')
    print(ohtani_ev[ev_cols].to_string(index=False))
else:
    print('Statcast EV/Barrels: Ohtani not found')

=== Ohtani spot-check ===

FanGraphs:
         Name Team   G  PA  HR  RBI   AVG  wRC+
Shohei Ohtani  LAD 158 727  55  102 0.282 172.0

Statcast EV/Barrels:
         Name  avg_hit_speed  max_hit_speed  brl_percent
Shohei Ohtani           94.9          120.0         23.5


In [136]:
# 3. Confirm top-300 pipeline output
print('=== Top-300 pipeline checks ===')
print(f'Total players:    {len(top_300)}')
print(f'Batter count:     {(top_300["player_type"] == "batter").sum()}')
print(f'Pitcher count:    {(top_300["player_type"] == "pitcher").sum()}')

# Check no duplicate players
dupes = top_300['Name'].duplicated().sum()
print(f'Duplicate names:  {dupes} (expected 0 or close to 0 for two-way players)')

# Check key stat completeness for batters only
bat_subset = top_300[top_300['player_type'] == 'batter']
core_bat = [c for c in ['HR', 'RBI', 'AVG', 'OBP'] if c in bat_subset.columns]
if core_bat:
    pct_complete = bat_subset[core_bat].notna().mean() * 100
    print('\nBatter stat completeness (%):')
    print(pct_complete.round(1).to_string())

=== Top-300 pipeline checks ===
Total players:    300
Batter count:     200
Pitcher count:    100
Duplicate names:  1 (expected 0 or close to 0 for two-way players)

Batter stat completeness (%):
HR     100.0
RBI    100.0
AVG    100.0
OBP    100.0


In [137]:
# 4. Confirm team data has all 30 MLB teams
print('=== Team data checks ===')
print(f'Team batting rows:  {len(team_bat)} (expected 30)')
print(f'Team pitching rows: {len(team_pit)} (expected 30)')

if 'Team' in team_bat.columns:
    print('\nAll teams present in batting data:')
    print(sorted(team_bat['Team'].tolist()))

=== Team data checks ===
Team batting rows:  30 (expected 30)
Team pitching rows: 30 (expected 30)

All teams present in batting data:
['ARI', 'ATH', 'ATL', 'BAL', 'BOS', 'CHC', 'CHW', 'CIN', 'CLE', 'COL', 'DET', 'HOU', 'KCR', 'LAA', 'LAD', 'MIA', 'MIL', 'MIN', 'NYM', 'NYY', 'PHI', 'PIT', 'SDP', 'SEA', 'SFG', 'STL', 'TBR', 'TEX', 'TOR', 'WSN']
